# Parte SHAP della pipeline blu



Installazione pacchetti

In [1]:

%pip install pandas openpyxl pyarrow scikit-learn matplotlib seaborn shap imbalanced-learn


In [2]:
# CARTELLA BASE DEI DATI E DEI RISULTATI
from pathlib import Path

# Impostare la cartella che contiene data/ e results/.
# Se BASE esiste gia' nella sessione, viene mantenuta.
if "BASE" not in globals():
    BASE = Path.cwd()


In [3]:
# LIBRERIE NECESSARIE ALLE FUNZIONI SHAP
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap


**Loader generico per progetto e snapshot**

In [4]:
# ============================================================
# LOADER GENERICO PER PROGETTO E SNAPSHOT
# ============================================================

VALID_SNAPSHOTS = [
    "T0",
    "T1C",
    "T24",
    "T72",
    "T7D",
    "FULL"
]


def load_project_dataset(
    base_dir,
    project_name,
    snapshot
):
    snapshot = snapshot.upper()

    if snapshot not in VALID_SNAPSHOTS:
        raise ValueError(
            f"Snapshot non valido: {snapshot}. "
            f"Valori ammessi: {VALID_SNAPSHOTS}"
        )

    file_path = (
        base_dir
        / "data"
        / "raw"
        / project_name
        / f"{project_name}_{snapshot}.csv"
    )

    if not file_path.exists():
        raise FileNotFoundError(
            f"Dataset non trovato:\n{file_path}"
        )

    return pd.read_csv(file_path)


print(
    "Loader generico configurato per "
    "T0, T1C, T24, T72, T7D e FULL."
)

Loader generico configurato per T0, T1C, T24, T72, T7D e FULL.


In [5]:
# ============================================================
# RUOLI DELLE COLONNE
# ============================================================

ID_COLUMNS = [
    "bug_id"
]

TARGET_COLUMN = "fixing_time"

TARGET_DERIVED_COLUMNS = [
    "fixing_time_hours",
    "fixing_time_days",
    "fixing_time_log",
    "residual_fixing_time_days",
    "residual_fixing_time_hours",
    "residual_fixing_time_log",
    "resolved_date"
]

METADATA_COLUMNS = [
    "creation_ts",
    "product",
    "component",
    "version",
    "rep_platform",
    "op_sys",
    "first_severity",
    "first_priority",
    "first_severity_status",
    "first_priority_status",
    "reporter",
    "assigned_to",
    "short_desc"
]



In [6]:
# ============================================================
# PREPARAZIONE SICURA DI X, y E bug_id
# ============================================================

def prepare_model_data(
    df,
    id_columns,
    target_column,
    target_derived_columns,
    metadata_columns
):
    excluded_columns = (
        id_columns
        + [target_column]
        + target_derived_columns
        + metadata_columns
    )

    predictor_columns = [
        column
        for column in df.columns
        if column not in excluded_columns
    ]

    X = df[predictor_columns].copy()
    y = df[target_column].copy()
    bug_ids = df["bug_id"].copy()

    return X, y, bug_ids, predictor_columns


print(
    "Funzione per preparare X, y e bug_id configurata."
)

Funzione per preparare X, y e bug_id configurata.


In [7]:
# ============================================================
# APPLICAZIONE DELLO SPLIT A UN DATAFRAME SNAPSHOT
# ============================================================

def apply_bug_id_split(
    df,
    dev_bug_ids,
    holdout_bug_ids,
    bug_id_column="bug_id"
):
    df_dev = df[
        df[bug_id_column].isin(dev_bug_ids)
    ].copy()

    df_holdout = df[
        df[bug_id_column].isin(holdout_bug_ids)
    ].copy()

    return df_dev, df_holdout


print(
    "Funzione per applicare lo split "
    "development/holdout agli snapshot configurata."
)

Funzione per applicare lo split development/holdout agli snapshot configurata.


In [8]:
# ============================================================
# PREPARAZIONE DEVELOPMENT / HOLDOUT PER UNO SNAPSHOT
# ============================================================

def prepare_snapshot_split(
    df,
    dev_bug_ids,
    holdout_bug_ids
):
    # Applica lo split stabile basato sui bug_id
    df_dev, df_holdout = apply_bug_id_split(
        df,
        dev_bug_ids,
        holdout_bug_ids
    )

    # Prepara i dati di development
    X_dev, y_dev, bug_ids_dev, predictor_columns = prepare_model_data(
        df=df_dev,
        id_columns=ID_COLUMNS,
        target_column=TARGET_COLUMN,
        target_derived_columns=TARGET_DERIVED_COLUMNS,
        metadata_columns=METADATA_COLUMNS
    )

    # Prepara i dati di holdout usando la stessa logica
    X_holdout, y_holdout, bug_ids_holdout, _ = prepare_model_data(
        df=df_holdout,
        id_columns=ID_COLUMNS,
        target_column=TARGET_COLUMN,
        target_derived_columns=TARGET_DERIVED_COLUMNS,
        metadata_columns=METADATA_COLUMNS
    )

    return {
        "df_dev": df_dev,
        "df_holdout": df_holdout,
        "X_dev": X_dev,
        "y_dev": y_dev,
        "bug_ids_dev": bug_ids_dev,
        "X_holdout": X_holdout,
        "y_holdout": y_holdout,
        "bug_ids_holdout": bug_ids_holdout,
        "predictor_columns": predictor_columns
    }


print(
    "Funzione per preparare development e holdout "
    "di uno snapshot configurata."
)

Funzione per preparare development e holdout di uno snapshot configurata.


In [9]:
# ============================================================
# CARICAMENTO DELLO SPLIT CONGELATO bug_id -> split
# ============================================================

def load_bug_id_split_assignment(
    base_dir,
    project_name
):
    split_path = (
        base_dir
        / "data"
        / "processed"
        / project_name
        / f"{project_name.lower()}_bug_id_split.csv"
    )

    if not split_path.exists():
        raise FileNotFoundError(
            f"Split congelato non trovato:\n{split_path}"
        )

    split_assignment = pd.read_csv(split_path)

    required_columns = {
        "bug_id",
        "split"
    }

    if not required_columns.issubset(
        split_assignment.columns
    ):
        raise ValueError(
            "Il file dello split congelato non contiene "
            "le colonne richieste: bug_id, split."
        )

    valid_split_values = {
        "development",
        "holdout"
    }

    observed_split_values = set(
        split_assignment["split"].dropna().unique()
    )

    if not observed_split_values.issubset(
        valid_split_values
    ):
        raise ValueError(
            "Valori non validi nella colonna split: "
            f"{observed_split_values}"
        )

    if split_assignment["bug_id"].duplicated().any():
        raise ValueError(
            "Sono presenti bug_id duplicati "
            "nel file dello split congelato."
        )

    dev_bug_ids = set(
        split_assignment.loc[
            split_assignment["split"] == "development",
            "bug_id"
        ]
    )

    holdout_bug_ids = set(
        split_assignment.loc[
            split_assignment["split"] == "holdout",
            "bug_id"
        ]
    )

    return (
        dev_bug_ids,
        holdout_bug_ids,
        split_assignment
    )


print(
    "Funzione per il caricamento dello split "
    "congelato configurata."
)

Funzione per il caricamento dello split congelato configurata.


### **7. INTERPRETABILITÀ E SHAP**

**Recupero delle feature dopo il preprocessing**

In [10]:

# ============================================================
# PARTE 7.1 - NOMI DELLE FEATURE DOPO IL PREPROCESSING
# ============================================================

import numpy as np


def get_features_after_preprocessing(
    fitted_pipeline,
    original_feature_names
):
    """
    Recupera i nomi delle feature effettivamente
    mantenute dopo il preprocessing.

    Gestisce:
    - preprocessing come step unico;
    - imputer, variance_filter e scaler
      come step separati.

    Restituisce una lista di nomi di feature.
    """

    names = np.asarray(
        original_feature_names,
        dtype=object
    )

    if "preprocessing" in fitted_pipeline.named_steps:

        preprocessor = (
            fitted_pipeline.named_steps["preprocessing"]
        )

        return list(
            preprocessor.get_feature_names_out(names)
        )

    for step_name in [
        "imputer",
        "variance_filter",
        "scaler"
    ]:

        if step_name in fitted_pipeline.named_steps:

            transformer = (
                fitted_pipeline.named_steps[step_name]
            )

            names = transformer.get_feature_names_out(
                names
            )

    return list(names)


print(
    "Recupero dei nomi delle feature dopo "
    "il preprocessing configurato."
)


Recupero dei nomi delle feature dopo il preprocessing configurato.


**Preparazione per SHAP**

In [11]:
# ============================================================
# 7.2 PREPARAZIONE DEI DATI PER SHAP
# VERSIONE COMPATIBILE CON REGRESSIONE E CLASSIFICAZIONE
# ============================================================

def transform_features_for_shap(
    fitted_pipeline,
    X
):
    X_transformed = X.copy()

    # --------------------------------------------------------
    # CASO 1: pipeline di regressione
    # preprocessing è contenuto nello step "preprocessing"
    # --------------------------------------------------------
    if "preprocessing" in fitted_pipeline.named_steps:

        preprocessing = fitted_pipeline.named_steps["preprocessing"]

        X_transformed = preprocessing.transform(
            X_transformed
        )

        # Lo scaler, se presente, è esterno al preprocessing
        if "scaler" in fitted_pipeline.named_steps:
            X_transformed = fitted_pipeline.named_steps[
                "scaler"
            ].transform(X_transformed)

    # --------------------------------------------------------
    # CASO 2: pipeline di classificazione
    # imputer, variance_filter e scaler sono step separati
    # --------------------------------------------------------
    else:

        if "imputer" in fitted_pipeline.named_steps:
            X_transformed = fitted_pipeline.named_steps[
                "imputer"
            ].transform(X_transformed)

        if "variance_filter" in fitted_pipeline.named_steps:
            X_transformed = fitted_pipeline.named_steps[
                "variance_filter"
            ].transform(X_transformed)

        if "scaler" in fitted_pipeline.named_steps:
            X_transformed = fitted_pipeline.named_steps[
                "scaler"
            ].transform(X_transformed)

    return X_transformed


print(
    "Funzione per trasformare i dati SHAP aggiornata "
    "per regressione e classificazione."
)

Funzione per trasformare i dati SHAP aggiornata per regressione e classificazione.


**Calcolo dei valori SHAP**

In [12]:

# ============================================================
# 7.3 CALCOLO SHAP
# BACKGROUND SEPARATO DAI DATI DA SPIEGARE
# ============================================================

import numpy as np
import shap


def compute_shap_values(
    fitted_pipeline,
    X_transformed,
    X_background_transformed,
    max_background=50,
    max_explain=30,
    random_state=42,
    return_indices=False
):
    """
    Calcola SHAP su un campione di osservazioni.

    X_background_transformed:
        dati provenienti dal DEVELOPMENT.

    X_transformed:
        osservazioni da spiegare, ad esempio dal TEST.

    Entrambi devono essere già trasformati usando
    il preprocessing del modello addestrato.

    random_state controlla sia il campionamento
    sia il seed dell'explainer.
    """

    model = fitted_pipeline.named_steps["model"]

    X_explain = np.asarray(X_transformed)

    X_background = np.asarray(
        X_background_transformed
    )

    # --------------------------------------------------------
    # 1. VALIDAZIONE
    # --------------------------------------------------------

    if X_explain.ndim != 2 or X_background.ndim != 2:
        raise ValueError(
            "SHAP richiede matrici bidimensionali."
        )

    if len(X_explain) == 0 or len(X_background) == 0:
        raise ValueError(
            "Background e dati da spiegare "
            "non possono essere vuoti."
        )

    if X_explain.shape[1] != X_background.shape[1]:
        raise ValueError(
            "Background e dati da spiegare devono "
            "avere le stesse feature."
        )

    if max_background < 1 or max_explain < 1:
        raise ValueError(
            "I limiti di campionamento devono "
            "essere positivi."
        )

    rng = np.random.default_rng(
        random_state
    )

    # --------------------------------------------------------
    # 2. CAMPIONAMENTO DEL BACKGROUND
    # --------------------------------------------------------

    idx_background = np.arange(
        len(X_background)
    )

    if len(X_background) > max_background:

        idx_background = rng.choice(
            len(X_background),
            size=max_background,
            replace=False
        )

        X_background = X_background[
            idx_background
        ]

    # --------------------------------------------------------
    # 3. CAMPIONAMENTO DELLE OSSERVAZIONI
    # --------------------------------------------------------

    idx_explain = np.arange(
        len(X_explain)
    )

    if len(X_explain) > max_explain:

        idx_explain = rng.choice(
            len(X_explain),
            size=max_explain,
            replace=False
        )

        X_explain = X_explain[
            idx_explain
        ]

    # --------------------------------------------------------
    # 4. SCELTA DELLA FUNZIONE DA SPIEGARE
    # --------------------------------------------------------

    if hasattr(model, "predict_proba"):

        # Classificazione: probabilità SLOW = 1.
        classes = np.asarray(
            model.classes_
        )

        if 1 not in classes:
            raise ValueError(
                "La classe SLOW=1 non è "
                "presente nel modello."
            )

        slow_index = int(
            np.flatnonzero(
                classes == 1
            )[0]
        )

        def prediction_function(X):
            return model.predict_proba(X)[
                :, slow_index
            ]

        explainer = shap.Explainer(
            prediction_function,
            X_background,
            algorithm="permutation",
            seed=random_state
        )

    elif hasattr(model, "decision_function"):

        # Classificazione senza predict_proba:
        # spieghiamo il decision score.
        classes = np.asarray(
            model.classes_
        )

        if not np.array_equal(
            classes,
            [0, 1]
        ):
            raise ValueError(
                "Verificare l'orientamento del "
                "decision score rispetto a SLOW=1."
            )

        explainer = shap.Explainer(
            model.decision_function,
            X_background,
            algorithm="permutation",
            seed=random_state
        )

    else:

        # Regressione: proviamo prima l'explainer
        # direttamente compatibile con il modello.
        try:

            explainer = shap.Explainer(
                model,
                X_background,
                seed=random_state
            )

        except (TypeError, ValueError):

            explainer = shap.Explainer(
                model.predict,
                X_background,
                algorithm="permutation",
                seed=random_state
            )

    # --------------------------------------------------------
    # 5. CALCOLO SHAP
    # --------------------------------------------------------

    shap_values = explainer(
        X_explain
    )

    values = np.asarray(
        shap_values.values
    )

    if values.ndim != 2:
        raise ValueError(
            f"Forma SHAP inattesa: {values.shape}. "
            "È attesa una matrice "
            "osservazioni x feature."
        )

    if values.shape != X_explain.shape:
        raise ValueError(
            "Dimensioni SHAP non coerenti "
            "con le osservazioni spiegate."
        )

    # --------------------------------------------------------
    # 6. RESTITUZIONE DEI RISULTATI
    # --------------------------------------------------------

    if return_indices:
        return (
            explainer,
            shap_values,
            idx_explain
        )

    return explainer, shap_values


print(
    "Funzione SHAP aggiornata con seed esplicito "
    "e background separato."
)


Funzione SHAP aggiornata con seed esplicito e background separato.


**Tabella ordinata dei valori SHAP**

In [13]:
# ============================================================
# 7.4 IMPORTANZA GLOBALE DELLE FEATURE
# ============================================================

def summarize_global_shap_importance(
    shap_values,
    feature_names
):
    values = np.asarray(shap_values.values)

    # Caso standard: una matrice [n_osservazioni, n_feature]
    if values.ndim == 2:
        mean_abs_shap = np.abs(values).mean(axis=0)

    # Caso con una dimensione aggiuntiva, ad esempio classificazione
    elif values.ndim == 3:
        mean_abs_shap = np.abs(values).mean(axis=(0, 2))

    else:
        raise ValueError(
            "Formato dei valori SHAP non riconosciuto."
        )

    if len(mean_abs_shap) != len(feature_names):
        raise ValueError(
            "Il numero di feature non coincide con "
            "il numero di valori SHAP aggregati."
        )

    shap_importance = pd.DataFrame({
        "feature": feature_names,
        "mean_abs_shap": mean_abs_shap
    })

    shap_importance = shap_importance.sort_values(
        by="mean_abs_shap",
        ascending=False
    ).reset_index(drop=True)

    return shap_importance


print(
    "Funzione per riassumere l'importanza globale SHAP configurata."
)

Funzione per riassumere l'importanza globale SHAP configurata.


**Spiegazione di una singola osservazione**

In [14]:
# ============================================================
# 7.5 SPIEGAZIONE LOCALE DI UNA SINGOLA OSSERVAZIONE
# VERSIONE COERENTE CON REGRESSIONE E CLASSE SLOW
# ============================================================

def get_local_shap_explanation(
    shap_values,
    feature_names,
    observation_index=0
):
    values = np.asarray(shap_values.values)

    # --------------------------------------------------------
    # Caso standard:
    # regressione oppure classificazione già ridotta allo
    # score continuo della classe SLOW = 1
    # --------------------------------------------------------
    if values.ndim == 2:
        local_values = values[observation_index]

    # --------------------------------------------------------
    # Caso eccezionale con più output SHAP
    # Manteniamo compatibilità scegliendo la classe positiva
    # --------------------------------------------------------
    elif values.ndim == 3:
        local_values = values[
            observation_index,
            :,
            -1
        ]

    else:
        raise ValueError(
            "Formato dei valori SHAP non riconosciuto."
        )

    if len(local_values) != len(feature_names):
        raise ValueError(
            "Il numero di feature non coincide con "
            "il numero di valori SHAP locali."
        )

    local_explanation = pd.DataFrame({
        "feature": feature_names,
        "shap_value": local_values
    })

    local_explanation["abs_shap_value"] = np.abs(
        local_explanation["shap_value"]
    )

    local_explanation = local_explanation.sort_values(
        by="abs_shap_value",
        ascending=False
    ).reset_index(drop=True)

    return local_explanation


print(
    "Funzione di spiegazione locale SHAP aggiornata "
    "per regressione e classe SLOW."
)

Funzione di spiegazione locale SHAP aggiornata per regressione e classe SLOW.


**Salvataggio degli output SHAP**

In [15]:

# ============================================================
# 7.6 SALVATAGGIO COMPLETO DEGLI OUTPUT SHAP
# VERSIONE MULTI-SCENARIO CON IDENTIFICATIVI DEI BUG
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


def save_shap_artifacts(
    base_dir,
    project,
    model_name,
    configuration,
    shap_values,
    feature_names,
    global_importance,
    snapshot,
    scenario,
    task,
    threshold=None,
    balancing=None,
    local_explanation=None,
    stability_results=None,
    explained_ids=None
):
    """
    Salva gli output SHAP di un esperimento.

    Supporta:
        - within_project, pooled, lopo
        - regressione e classificazione
        - identificativi delle osservazioni spiegate
        - ricostruzione dell'oggetto shap.Explanation

    Non sovrascrive esperimenti precedenti.
    """

    # --------------------------------------------------------
    # 1. NORMALIZZAZIONE DEI NOMI
    # --------------------------------------------------------

    def slugify(value):

        if value is None:
            return None

        return (
            str(value)
            .strip()
            .lower()
            .replace(" ", "_")
            .replace("/", "_")
        )

    project_slug = slugify(project) or "all_projects"
    model_slug = slugify(model_name)
    config_slug = slugify(configuration)
    snapshot_slug = slugify(snapshot)
    scenario_slug = slugify(scenario)
    task_slug = slugify(task)
    threshold_slug = slugify(threshold)
    balancing_slug = slugify(balancing)

    # --------------------------------------------------------
    # 2. IDENTIFICATIVO DELL'ESPERIMENTO
    # --------------------------------------------------------

    stem_parts = [
        project_slug,
        snapshot_slug,
        scenario_slug,
        task_slug,
        model_slug,
        config_slug
    ]

    if threshold_slug is not None:
        stem_parts.append(threshold_slug)

    if balancing_slug is not None:
        stem_parts.append(balancing_slug)

    stem = "_".join(stem_parts)

    # --------------------------------------------------------
    # 3. VALIDAZIONE DEI VALORI SHAP
    # --------------------------------------------------------

    values = np.asarray(
        shap_values.values
    )

    feature_names = list(feature_names)

    if values.ndim != 2:
        raise ValueError(
            "I valori SHAP devono avere dimensioni "
            "(n_osservazioni, n_feature)."
        )

    n_observations, n_features = values.shape

    if n_observations == 0 or n_features == 0:
        raise ValueError(
            "I valori SHAP non possono essere vuoti."
        )

    if len(feature_names) != n_features:
        raise ValueError(
            "Il numero di feature non corrisponde "
            "alle colonne dei valori SHAP."
        )

    if not np.isfinite(values).all():
        raise ValueError(
            "I valori SHAP contengono NaN o infiniti."
        )

    # --------------------------------------------------------
    # 4. VALIDAZIONE BASE VALUES E DATI SPIEGATI
    # --------------------------------------------------------

    if (
        shap_values.base_values is None
        or shap_values.data is None
    ):
        raise ValueError(
            "L'oggetto SHAP deve contenere "
            "base_values e data."
        )

    base_values = np.asarray(
        shap_values.base_values
    )

    explained_data = np.asarray(
        shap_values.data
    )

    if explained_data.shape != values.shape:
        raise ValueError(
            "I dati spiegati non hanno "
            "le stesse dimensioni dei valori SHAP."
        )

    if base_values.ndim == 0:
        base_values = np.repeat(
            base_values,
            n_observations
        )

    if (
        base_values.ndim != 1
        or len(base_values) != n_observations
    ):
        raise ValueError(
            "base_values deve contenere un valore "
            "per osservazione spiegata."
        )

    if (
        not np.isfinite(base_values).all()
        or not np.isfinite(
            explained_data
        ).all()
    ):
        raise ValueError(
            "base_values o data contengono "
            "valori non finiti."
        )

    # --------------------------------------------------------
    # 5. VALIDAZIONE IMPORTANZA GLOBALE
    # --------------------------------------------------------

    if not isinstance(
        global_importance,
        pd.DataFrame
    ):
        raise TypeError(
            "global_importance deve essere un DataFrame."
        )

    required_importance_columns = [
        "feature",
        "mean_abs_shap"
    ]

    if not all(
        col in global_importance.columns
        for col in required_importance_columns
    ):
        raise ValueError(
            "global_importance deve contenere "
            "feature e mean_abs_shap."
        )

    if (
        len(global_importance) != n_features
        or set(global_importance["feature"])
        != set(feature_names)
        or global_importance[
            "feature"
        ].duplicated().any()
    ):
        raise ValueError(
            "Le feature di global_importance "
            "non corrispondono alle feature SHAP."
        )

    # --------------------------------------------------------
    # 6. VALIDAZIONE IDENTIFICATIVI DEI BUG
    # --------------------------------------------------------

    if explained_ids is not None:

        required_id_columns = [
            "project",
            "bug_id"
        ]

        if not isinstance(
            explained_ids,
            pd.DataFrame
        ):
            raise TypeError(
                "explained_ids deve essere un DataFrame."
            )

        if not all(
            col in explained_ids.columns
            for col in required_id_columns
        ):
            raise ValueError(
                "explained_ids deve contenere "
                "project e bug_id."
            )

        if len(explained_ids) != n_observations:
            raise ValueError(
                "Il numero di identificativi non "
                "corrisponde alle righe SHAP."
            )

        if explained_ids[
            required_id_columns
        ].isna().any().any():
            raise ValueError(
                "Gli identificativi SHAP "
                "contengono valori mancanti."
            )

        if explained_ids.duplicated(
            required_id_columns
        ).any():
            raise ValueError(
                "Gli identificativi SHAP "
                "contengono duplicati."
            )

    # --------------------------------------------------------
    # 7. CARTELLA DELL'ESPERIMENTO
    # --------------------------------------------------------

    shap_dir = (
        Path(base_dir)
        / "results"
        / "shap"
        / stem
    )

    if shap_dir.exists():
        raise FileExistsError(
            f"Output SHAP già esistenti: {shap_dir}"
        )

    shap_dir.mkdir(
        parents=True,
        exist_ok=False
    )

    # --------------------------------------------------------
    # 8. PERCORSI DEGLI OUTPUT
    # --------------------------------------------------------

    global_path = (
        shap_dir
        / f"{stem}_global_importance.csv"
    )

    values_path = (
        shap_dir
        / f"{stem}_shap_values.npy"
    )

    base_values_path = (
        shap_dir
        / f"{stem}_base_values.npy"
    )

    data_path = (
        shap_dir
        / f"{stem}_explained_data.npy"
    )

    features_path = (
        shap_dir
        / f"{stem}_feature_names.csv"
    )

    # --------------------------------------------------------
    # 9. SALVATAGGIO DEGLI OUTPUT PRINCIPALI
    # --------------------------------------------------------

    global_importance.to_csv(
        global_path,
        index=False
    )

    np.save(
        values_path,
        values
    )

    np.save(
        base_values_path,
        base_values
    )

    np.save(
        data_path,
        explained_data
    )

    pd.DataFrame({
        "feature": feature_names
    }).to_csv(
        features_path,
        index=False
    )

    # --------------------------------------------------------
    # 10. IDENTIFICATIVI DELLE OSSERVAZIONI SPIEGATE
    # --------------------------------------------------------

    explained_ids_path = None

    if explained_ids is not None:

        explained_ids_path = (
            shap_dir
            / f"{stem}_explained_ids.csv"
        )

        explained_ids[
            ["project", "bug_id"]
        ].reset_index(
            drop=True
        ).to_csv(
            explained_ids_path,
            index=False
        )

    # --------------------------------------------------------
    # 11. SPIEGAZIONE LOCALE
    # --------------------------------------------------------

    local_path = None

    if local_explanation is not None:

        local_path = (
            shap_dir
            / f"{stem}_local_explanation.csv"
        )

        local_explanation.to_csv(
            local_path,
            index=False
        )

    # --------------------------------------------------------
    # 12. RISULTATI DI STABILITÀ
    # --------------------------------------------------------

    stability_path = None

    if stability_results is not None:

        stability_path = (
            shap_dir
            / f"{stem}_stability_results.csv"
        )

        if isinstance(
            stability_results,
            dict
        ):

            pd.DataFrame([
                stability_results
            ]).to_csv(
                stability_path,
                index=False
            )

        else:

            stability_results.to_csv(
                stability_path,
                index=False
            )

    # --------------------------------------------------------
    # 13. RESTITUZIONE DEI PERCORSI
    # --------------------------------------------------------

    return {
        "global_importance_path": global_path,
        "shap_values_path": values_path,
        "base_values_path": base_values_path,
        "data_path": data_path,
        "feature_names_path": features_path,
        "explained_ids_path": explained_ids_path,
        "local_explanation_path": local_path,
        "stability_results_path": stability_path,
    }


print(
    "Salvataggio SHAP aggiornato con "
    "base_values, data e identificativi dei bug."
)


Salvataggio SHAP aggiornato con base_values, data e identificativi dei bug.


**Stabilità delle spiegazioni SHAP**

In [16]:
# ============================================================
# 7.7 STABILITÀ DELLE IMPORTANZE SHAP
# ============================================================

def summarize_shap_stability(
    importance_tables
):
    """
    importance_tables:
        dizionario nel formato
        {
            "run_1": dataframe_importanza,
            "run_2": dataframe_importanza,
            ...
        }
    """

    merged = None

    for run_name, importance_df in importance_tables.items():

        current = importance_df[
            ["feature", "mean_abs_shap"]
        ].copy()

        current = current.rename(
            columns={
                "mean_abs_shap": f"importance_{run_name}"
            }
        )

        current[f"rank_{run_name}"] = current[
            f"importance_{run_name}"
        ].rank(
            ascending=False,
            method="average"
        )

        if merged is None:
            merged = current
        else:
            merged = merged.merge(
                current,
                on="feature",
                how="outer"
            )

    rank_columns = [
        col
        for col in merged.columns
        if col.startswith("rank_")
    ]

    merged["mean_rank"] = merged[
        rank_columns
    ].mean(axis=1)

    merged["rank_std"] = merged[
        rank_columns
    ].std(axis=1)

    merged = merged.sort_values(
        by="mean_rank",
        ascending=True
    ).reset_index(drop=True)

    return merged


print(
    "Funzione per analizzare la stabilità delle "
    "importanze SHAP configurata."
)

Funzione per analizzare la stabilità delle importanze SHAP configurata.


**Misure di stabilità: Spearman e Jaccard**

In [17]:
# ============================================================
# 7.8 STABILITÀ QUANTITATIVA DEI RANKING SHAP
# Spearman rho + Jaccard@5 + Jaccard@10
# ============================================================

from scipy.stats import spearmanr


def compare_shap_rankings(
    importance_a,
    importance_b,
    top_k_values=(5, 10)
):
    """
    Confronta due ranking SHAP.
    """

    a = importance_a[
        ["feature", "mean_abs_shap"]
    ].copy()

    b = importance_b[
        ["feature", "mean_abs_shap"]
    ].copy()

    # Rank: 1 = feature più importante
    a["rank_a"] = a["mean_abs_shap"].rank(
        ascending=False,
        method="average"
    )

    b["rank_b"] = b["mean_abs_shap"].rank(
        ascending=False,
        method="average"
    )

    # Confrontiamo solo le feature presenti in entrambe
    merged = a[
        ["feature", "rank_a"]
    ].merge(
        b[
            ["feature", "rank_b"]
        ],
        on="feature",
        how="inner"
    )

    # Spearman
    if len(merged) < 2:
        spearman_rho = np.nan
        spearman_pvalue = np.nan
    else:
        spearman_rho, spearman_pvalue = spearmanr(
            merged["rank_a"],
            merged["rank_b"]
        )

    risultati = {
        "spearman_rho": spearman_rho,
        "spearman_pvalue": spearman_pvalue
    }

    # Jaccard@k
    for k in top_k_values:

        top_a = set(
            a.sort_values(
                by="mean_abs_shap",
                ascending=False
            ).head(k)["feature"]
        )

        top_b = set(
            b.sort_values(
                by="mean_abs_shap",
                ascending=False
            ).head(k)["feature"]
        )

        union = top_a | top_b
        intersection = top_a & top_b

        if len(union) == 0:
            jaccard = np.nan
        else:
            jaccard = len(intersection) / len(union)

        risultati[f"jaccard_at_{k}"] = jaccard

    return risultati


def build_shap_stability_matrices(
    importance_tables
):
    """
    Costruisce matrici complete di stabilità SHAP.

    importance_tables:
        dizionario nel formato

        {
            "run_1": dataframe_importanza,
            "run_2": dataframe_importanza,
            ...
        }
    """

    run_names = list(importance_tables.keys())

    spearman_matrix = pd.DataFrame(
        np.nan,
        index=run_names,
        columns=run_names
    )

    jaccard5_matrix = pd.DataFrame(
        np.nan,
        index=run_names,
        columns=run_names
    )

    jaccard10_matrix = pd.DataFrame(
        np.nan,
        index=run_names,
        columns=run_names
    )

    for run_a in run_names:
        for run_b in run_names:

            comparison = compare_shap_rankings(
                importance_a=importance_tables[run_a],
                importance_b=importance_tables[run_b],
                top_k_values=(5, 10)
            )

            spearman_matrix.loc[
                run_a,
                run_b
            ] = comparison["spearman_rho"]

            jaccard5_matrix.loc[
                run_a,
                run_b
            ] = comparison["jaccard_at_5"]

            jaccard10_matrix.loc[
                run_a,
                run_b
            ] = comparison["jaccard_at_10"]

    return {
        "spearman": spearman_matrix,
        "jaccard_at_5": jaccard5_matrix,
        "jaccard_at_10": jaccard10_matrix
    }


print(
    "Funzioni per confrontare e costruire le matrici "
    "di stabilità SHAP configurate."
)

Funzioni per confrontare e costruire le matrici di stabilità SHAP configurate.


**Visualizzazione delle matrici**

In [18]:
# ============================================================
# 7.8 VISUALIZZAZIONE DELLE MATRICI DI STABILITÀ SHAP
# ============================================================

import matplotlib.pyplot as plt


def plot_shap_stability_matrix(
    matrix,
    title
):
    plt.figure(
        figsize=(8, 6)
    )

    plt.imshow(
        matrix.astype(float)
    )

    plt.xticks(
        range(len(matrix.columns)),
        matrix.columns,
        rotation=45,
        ha="right"
    )

    plt.yticks(
        range(len(matrix.index)),
        matrix.index
    )

    plt.colorbar(
        label="Similarity"
    )

    plt.title(title)
    plt.tight_layout()
    plt.show()


print(
    "Funzione per visualizzare le matrici "
    "di stabilità SHAP configurata."
)

Funzione per visualizzare le matrici di stabilità SHAP configurata.


**Salvataggio delle matrici**

In [19]:
# ============================================================
# 7.8 SALVATAGGIO DELLE MATRICI E HEATMAP DI STABILITÀ SHAP
# ============================================================

def save_shap_stability_outputs(
    base_dir,
    project,
    comparison_name,
    stability_matrices
):
    shap_dir = base_dir / "results" / "shap"
    shap_dir.mkdir(parents=True, exist_ok=True)

    project_slug = project.lower().replace(" ", "_")
    comparison_slug = comparison_name.lower().replace(" ", "_")

    saved_paths = {}

    titles = {
        "spearman": "SHAP Stability - Spearman",
        "jaccard_at_5": "SHAP Stability - Jaccard@5",
        "jaccard_at_10": "SHAP Stability - Jaccard@10"
    }

    for metric_name, matrix in stability_matrices.items():

        # -----------------------------
        # Salvataggio matrice CSV
        # -----------------------------
        csv_path = (
            shap_dir /
            f"{project_slug}_{comparison_slug}_{metric_name}.csv"
        )

        matrix.to_csv(csv_path)

        # -----------------------------
        # Salvataggio heatmap
        # -----------------------------
        figure_path = (
            shap_dir /
            f"{project_slug}_{comparison_slug}_{metric_name}_heatmap.png"
        )

        plt.figure(figsize=(8, 6))

        plt.imshow(
            matrix.astype(float)
        )

        plt.xticks(
            range(len(matrix.columns)),
            matrix.columns,
            rotation=45,
            ha="right"
        )

        plt.yticks(
            range(len(matrix.index)),
            matrix.index
        )

        plt.colorbar(
            label="Similarity"
        )

        plt.title(
            titles.get(metric_name, metric_name)
        )

        plt.tight_layout()
        plt.savefig(
            figure_path,
            dpi=300,
            bbox_inches="tight"
        )
        plt.close()

        saved_paths[f"{metric_name}_csv"] = csv_path
        saved_paths[f"{metric_name}_heatmap"] = figure_path

    return saved_paths


print(
    "Funzione per salvare matrici e heatmap "
    "di stabilità SHAP configurata."
)

Funzione per salvare matrici e heatmap di stabilità SHAP configurata.


### **8. Gestione degli scenari sperimentali: Within-project, Pooled e LOPO**

**Configurazione scenari**

In [20]:

# ============================================================
# 8.1 CONFIGURAZIONE DEGLI SCENARI SPERIMENTALI
# ============================================================

PROJECTS = [
    "Eclipse",
    "KDE",
    "Mozilla",
    "Novell",
    "OpenOffice",
    "OpenXchange",
    "W3C"
]

SNAPSHOTS = [
    "T0",
    "T1C",
    "T24",
    "T72",
    "T7D",
    "FULL"
]

SCENARIOS = [
    "within_project",
    "pooled",
    "lopo"
]

print("Progetti configurati:", len(PROJECTS))
print("Snapshot configurati:", len(SNAPSHOTS))
print("Scenari configurati:", SCENARIOS)


Progetti configurati: 7
Snapshot configurati: 6
Scenari configurati: ['within_project', 'pooled', 'lopo']


**Costruzione Pooled**

In [21]:

# ============================================================
# 8.4 COSTRUZIONE DEL DATASET POOLED
# CON SPLIT CONGELATI PER PROGETTO
# ============================================================

def build_pooled_dataset(
    base_dir,
    projects,
    snapshot
):

    development_parts = []
    holdout_parts = []
    feature_sets = []

    for project in projects:

        # Caricamento del dataset dello snapshot
        df = load_project_dataset(
            base_dir,
            project,
            snapshot
        )

        # Recupero dello split congelato
        dev_ids, holdout_ids, _ = (
            load_bug_id_split_assignment(
                base_dir,
                project
            )
        )

        # Controlli sugli identificativi
        if df["bug_id"].isna().any():
            raise ValueError(
                f"{project}: bug_id mancanti."
            )

        if df["bug_id"].duplicated().any():
            raise ValueError(
                f"{project}: bug_id duplicati."
            )

        if not set(df["bug_id"]).issubset(
            dev_ids | holdout_ids
        ):
            raise ValueError(
                f"{project}: bug_id non assegnati allo split."
            )

        # Riutilizzo della funzione della Parte 1
        prepared = prepare_snapshot_split(
            df,
            dev_ids,
            holdout_ids
        )

        # Conserviamo la provenienza dei bug
        # senza usarla come feature predittiva
        for split_name, parts in [
            ("dev", development_parts),
            ("holdout", holdout_parts)
        ]:

            X = prepared[f"X_{split_name}"].copy()
            y = prepared[f"y_{split_name}"].copy()
            bug_ids = prepared[
                f"bug_ids_{split_name}"
            ].copy()

            part = X.copy()

            # Colonne di tracciabilità aggiunte
            # soltanto dopo la separazione di X
            part["__project"] = project
            part["__bug_id"] = list(bug_ids)
            part["__target"] = list(y)

            parts.append(part)

        # Feature individuate sul development,
        # non tramite analisi dell'holdout
        feature_sets.append(
            set(prepared["X_dev"].columns)
        )

    # Feature comuni ai progetti
    common_features = set.intersection(
        *feature_sets
    )

    if not common_features:
        raise ValueError(
            "Nessuna feature comune tra i progetti."
        )

    # Ordine stabile e riproducibile
    common_features = sorted(common_features)

    # Unione degli insiemi
    df_dev_pooled = pd.concat(
        development_parts,
        ignore_index=True
    )

    df_holdout_pooled = pd.concat(
        holdout_parts,
        ignore_index=True
    )

    # Matrici predittive: soltanto feature
    X_dev_pooled = df_dev_pooled[
        common_features
    ].copy()

    X_holdout_pooled = df_holdout_pooled[
        common_features
    ].copy()

    # Target continuo
    y_dev_pooled = df_dev_pooled[
        "__target"
    ].copy()

    y_holdout_pooled = df_holdout_pooled[
        "__target"
    ].copy()

    # Identificativi composti:
    # progetto + bug_id
    ids_dev = df_dev_pooled[
        ["__project", "__bug_id"]
    ].copy()

    ids_holdout = df_holdout_pooled[
        ["__project", "__bug_id"]
    ].copy()

    # Verifica assenza duplicazioni e overlap
    dev_keys = set(
        map(tuple, ids_dev.to_numpy())
    )

    holdout_keys = set(
        map(tuple, ids_holdout.to_numpy())
    )

    if len(dev_keys) != len(ids_dev):
        raise ValueError(
            "Bug duplicati nel development pooled."
        )

    if len(holdout_keys) != len(ids_holdout):
        raise ValueError(
            "Bug duplicati nell'holdout pooled."
        )

    if dev_keys & holdout_keys:
        raise ValueError(
            "Overlap tra development e holdout pooled."
        )

    return {
        "X_dev": X_dev_pooled,
        "y_dev": y_dev_pooled,
        "ids_dev": ids_dev,
        "X_holdout": X_holdout_pooled,
        "y_holdout": y_holdout_pooled,
        "ids_holdout": ids_holdout,
        "feature_names": common_features,
        "snapshot": snapshot,
        "scenario": "pooled"
    }


print("Funzione pooled configurata.")


Funzione pooled configurata.


**Funzione LOPO**

In [22]:

# ============================================================
# 8.7 COSTRUZIONE DATASET LOPO
# LEAVE-ONE-PROJECT-OUT
# ============================================================

def build_lopo_dataset(
    base_dir,
    projects,
    excluded_project,
    snapshot
):

    if excluded_project not in projects:
        raise ValueError(
            "Il progetto escluso non è nella lista."
        )

    if len(projects) < 2:
        raise ValueError(
            "LOPO richiede almeno due progetti."
        )

    if len(set(projects)) != len(projects):
        raise ValueError(
            "La lista dei progetti contiene duplicati."
        )

    training_projects = [
        project
        for project in projects
        if project != excluded_project
    ]

    X_train_parts = []
    y_train_parts = []
    ids_train_parts = []
    training_feature_sets = []

    # --------------------------------------------------------
    # 1. CARICAMENTO DEI SOLI PROGETTI DI TRAINING
    # --------------------------------------------------------

    for project in training_projects:

        df = load_project_dataset(
            base_dir,
            project,
            snapshot
        )

        if (
            df["bug_id"].isna().any()
            or df["bug_id"].duplicated().any()
        ):
            raise ValueError(
                f"{project}: bug_id non validi."
            )

        X, y, bug_ids, feature_names = (
            prepare_model_data(
                df=df,
                id_columns=ID_COLUMNS,
                target_column=TARGET_COLUMN,
                target_derived_columns=TARGET_DERIVED_COLUMNS,
                metadata_columns=METADATA_COLUMNS
            )
        )

        X_train_parts.append(
            X.reset_index(drop=True)
        )

        y_train_parts.append(
            y.reset_index(drop=True)
        )

        ids_train_parts.append(
            pd.DataFrame({
                "project": project,
                "bug_id": list(bug_ids)
            })
        )

        training_feature_sets.append(
            set(feature_names)
        )

    # --------------------------------------------------------
    # 2. FEATURE COMUNI SOLO AI PROGETTI DI TRAINING
    # --------------------------------------------------------

    common_features = sorted(
        set.intersection(*training_feature_sets)
    )

    if not common_features:
        raise ValueError(
            "Nessuna feature comune ai progetti di training."
        )

    # --------------------------------------------------------
    # 3. COSTRUZIONE DEVELOPMENT LOPO
    # --------------------------------------------------------

    X_train = pd.concat(
        [
            X[common_features]
            for X in X_train_parts
        ],
        ignore_index=True
    )

    y_train = pd.concat(
        y_train_parts,
        ignore_index=True
    )

    ids_train = pd.concat(
        ids_train_parts,
        ignore_index=True
    )

    # --------------------------------------------------------
    # 4. CARICAMENTO DEL PROGETTO ESCLUSO
    # --------------------------------------------------------

    df_test = load_project_dataset(
        base_dir,
        excluded_project,
        snapshot
    )

    if (
        df_test["bug_id"].isna().any()
        or df_test["bug_id"].duplicated().any()
    ):
        raise ValueError(
            f"{excluded_project}: bug_id non validi."
        )

    X_test, y_test, bug_ids_test, _ = (
        prepare_model_data(
            df=df_test,
            id_columns=ID_COLUMNS,
            target_column=TARGET_COLUMN,
            target_derived_columns=TARGET_DERIVED_COLUMNS,
            metadata_columns=METADATA_COLUMNS
        )
    )

    # Il progetto di test NON determina
    # quali feature utilizzare.
    missing_features = (
        set(common_features) - set(X_test.columns)
    )

    if missing_features:
        raise ValueError(
            "Feature di training mancanti nel test: "
            f"{sorted(missing_features)}"
        )

    X_test = (
        X_test[common_features]
        .reset_index(drop=True)
    )

    y_test = y_test.reset_index(drop=True)

    ids_test = pd.DataFrame({
        "project": excluded_project,
        "bug_id": list(bug_ids_test)
    })

    # --------------------------------------------------------
    # 5. CONTROLLI ANTI-LEAKAGE
    # --------------------------------------------------------

    assert excluded_project not in set(
        ids_train["project"]
    )

    assert set(ids_test["project"]) == {
        excluded_project
    }

    assert X_train.columns.equals(X_test.columns)

    assert len(X_train) == len(y_train)
    assert len(X_test) == len(y_test)

    assert not ids_train.duplicated(
        ["project", "bug_id"]
    ).any()

    assert not ids_test.duplicated(
        ["project", "bug_id"]
    ).any()

    # --------------------------------------------------------
    # 6. RISULTATO
    # --------------------------------------------------------

    return {
        "X_dev": X_train,
        "y_dev": y_train,
        "ids_dev": ids_train,
        "X_test": X_test,
        "y_test": y_test,
        "ids_test": ids_test,
        "feature_names": common_features,
        "training_projects": training_projects,
        "excluded_project": excluded_project,
        "snapshot": snapshot,
        "scenario": "lopo"
    }


print("Funzione LOPO configurata.")


Funzione LOPO configurata.


**Interfaccia unica**

In [23]:

# ============================================================
# 8.11 INTERFACCIA UNICA PER I TRE SCENARI
# WITHIN-PROJECT / POOLED / LOPO
# ============================================================

def prepare_scenario_data(
    base_dir,
    scenario,
    snapshot,
    projects,
    project=None
):

    # --------------------------------------------------------
    # 1. WITHIN-PROJECT
    # --------------------------------------------------------

    if scenario == "within_project":

        if project is None:
            raise ValueError(
                "Within-project richiede un progetto."
            )

        if project not in projects:
            raise ValueError(
                f"Progetto non riconosciuto: {project}"
            )

        df = load_project_dataset(
            base_dir,
            project,
            snapshot
        )

        dev_ids, holdout_ids, _ = (
            load_bug_id_split_assignment(
                base_dir,
                project
            )
        )

        prepared = prepare_snapshot_split(
            df,
            dev_ids,
            holdout_ids
        )

        X_dev = prepared["X_dev"]
        y_dev = prepared["y_dev"]

        X_test = prepared["X_holdout"]
        y_test = prepared["y_holdout"]

        ids_dev = pd.DataFrame({
            "project": project,
            "bug_id": list(prepared["bug_ids_dev"])
        })

        ids_test = pd.DataFrame({
            "project": project,
            "bug_id": list(prepared["bug_ids_holdout"])
        })

    # --------------------------------------------------------
    # 2. POOLED
    # --------------------------------------------------------

    elif scenario == "pooled":

        prepared = build_pooled_dataset(
            base_dir=base_dir,
            projects=projects,
            snapshot=snapshot
        )

        X_dev = prepared["X_dev"]
        y_dev = prepared["y_dev"]

        X_test = prepared["X_holdout"]
        y_test = prepared["y_holdout"]

        ids_dev = prepared["ids_dev"].copy()
        ids_test = prepared["ids_holdout"].copy()

        ids_dev.columns = ["project", "bug_id"]
        ids_test.columns = ["project", "bug_id"]

    # --------------------------------------------------------
    # 3. LOPO
    # --------------------------------------------------------

    elif scenario == "lopo":

        if project is None:
            raise ValueError(
                "LOPO richiede il progetto escluso."
            )

        prepared = build_lopo_dataset(
            base_dir=base_dir,
            projects=projects,
            excluded_project=project,
            snapshot=snapshot
        )

        X_dev = prepared["X_dev"]
        y_dev = prepared["y_dev"]

        X_test = prepared["X_test"]
        y_test = prepared["y_test"]

        ids_dev = prepared["ids_dev"]
        ids_test = prepared["ids_test"]

    else:

        raise ValueError(
            f"Scenario non riconosciuto: {scenario}"
        )

    # --------------------------------------------------------
    # 4. ALLINEAMENTO DEGLI INDICI
    # --------------------------------------------------------

    X_dev = X_dev.reset_index(drop=True).copy()
    y_dev = y_dev.reset_index(drop=True).copy()
    ids_dev = ids_dev.reset_index(drop=True).copy()

    X_test = X_test.reset_index(drop=True).copy()
    y_test = y_test.reset_index(drop=True).copy()
    ids_test = ids_test.reset_index(drop=True).copy()

    # --------------------------------------------------------
    # 5. CONTROLLI STRUTTURALI
    # --------------------------------------------------------

    if not X_dev.columns.equals(X_test.columns):
        raise ValueError(
            "Feature non allineate tra development e test."
        )

    if not (
        len(X_dev) == len(y_dev) == len(ids_dev)
    ):
        raise ValueError(
            "Lunghezze development incoerenti."
        )

    if not (
        len(X_test) == len(y_test) == len(ids_test)
    ):
        raise ValueError(
            "Lunghezze test incoerenti."
        )

    if X_dev.empty or X_test.empty:
        raise ValueError(
            "Development o test vuoto."
        )

    if ids_dev.duplicated(
        ["project", "bug_id"]
    ).any():
        raise ValueError(
            "Bug duplicati nel development."
        )

    if ids_test.duplicated(
        ["project", "bug_id"]
    ).any():
        raise ValueError(
            "Bug duplicati nel test."
        )

    # Controllo assenza di overlap
    dev_keys = set(
        map(tuple, ids_dev[["project", "bug_id"]].to_numpy())
    )

    test_keys = set(
        map(tuple, ids_test[["project", "bug_id"]].to_numpy())
    )

    if dev_keys & test_keys:
        raise ValueError(
            "Overlap tra development e test."
        )

    # Controllo specifico LOPO
    if scenario == "lopo":

        if project in set(ids_dev["project"]):
            raise ValueError(
                "Progetto escluso presente nel development."
            )

        if set(ids_test["project"]) != {project}:
            raise ValueError(
                "Test LOPO contiene progetti non previsti."
            )

    # --------------------------------------------------------
    # 6. RESTITUZIONE DEI DATI
    # --------------------------------------------------------

    return {
        "scenario": scenario,
        "snapshot": snapshot,
        "project": project,

        "X_dev": X_dev,
        "y_dev": y_dev,
        "ids_dev": ids_dev,

        "X_test": X_test,
        "y_test": y_test,
        "ids_test": ids_test,

        "feature_names": list(X_dev.columns)
    }


print("Interfaccia unica dei tre scenari configurata.")


Interfaccia unica dei tre scenari configurata.


### **9. Integrazione SHAP e Explainability**

**Calcolo shap su modello già addestrato**

In [24]:

# ============================================================
# 9.1 ORCHESTRAZIONE SHAP MULTI-SCENARIO
# CON METADATI E BACKGROUND RIPRODUCIBILE
# ============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd
import shap


def run_scenario_shap(
    fitted_pipeline,
    scenario_data,
    base_dir,
    scenario,
    snapshot,
    project,
    task,
    model_name,
    configuration,
    threshold=None,
    balancing=None,
    max_background=50,
    max_explain=30,
    random_state=42
):
    """
    Calcola e salva SHAP per un modello già addestrato.

    Background: development.
    Osservazioni spiegate: test.

    Salva:
        - valori SHAP e importanze
        - base_values e dati spiegati
        - identificativi dei bug spiegati
        - identificativi del background
        - matrice numerica del background
        - metadati dell'esperimento

    Non riaddestra modelli.
    """

    # --------------------------------------------------------
    # 1. RECUPERO E VALIDAZIONE DEI DATI
    # --------------------------------------------------------

    X_dev = scenario_data["X_dev"]
    X_test = scenario_data["X_test"]

    ids_dev = scenario_data["ids_dev"]
    ids_test = scenario_data["ids_test"]

    if not X_dev.columns.equals(X_test.columns):
        raise ValueError(
            "Le feature di development e test "
            "non sono allineate."
        )

    if (
        len(X_dev) != len(ids_dev)
        or len(X_test) != len(ids_test)
    ):
        raise ValueError(
            "Le osservazioni e gli identificativi "
            "non sono allineati."
        )

    if len(X_dev) == 0 or len(X_test) == 0:
        raise ValueError(
            "Development e test non possono essere vuoti."
        )

    if max_background < 1 or max_explain < 1:
        raise ValueError(
            "Le dimensioni dei campioni devono "
            "essere positive."
        )

    rng = np.random.default_rng(
        random_state
    )

    # --------------------------------------------------------
    # 2. BACKGROUND DAL DEVELOPMENT
    # --------------------------------------------------------

    background_idx = rng.choice(
        len(X_dev),
        size=min(
            max_background,
            len(X_dev)
        ),
        replace=False
    )

    X_background_raw = X_dev.iloc[
        background_idx
    ].copy()

    background_ids = (
        ids_dev.iloc[background_idx][
            ["project", "bug_id"]
        ]
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # 3. OSSERVAZIONI DA SPIEGARE DAL TEST
    # --------------------------------------------------------

    explain_idx = rng.choice(
        len(X_test),
        size=min(
            max_explain,
            len(X_test)
        ),
        replace=False
    )

    X_explain_raw = X_test.iloc[
        explain_idx
    ].copy()

    explained_ids = (
        ids_test.iloc[explain_idx][
            ["project", "bug_id"]
        ]
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # 4. PREPROCESSING DEL MODELLO GIÀ FITTATO
    # --------------------------------------------------------

    X_background = transform_features_for_shap(
        fitted_pipeline,
        X_background_raw
    )

    X_explain = transform_features_for_shap(
        fitted_pipeline,
        X_explain_raw
    )

    feature_names = get_features_after_preprocessing(
        fitted_pipeline,
        list(X_dev.columns)
    )

    if (
        X_background.shape[1] != len(feature_names)
        or X_explain.shape[1] != len(feature_names)
    ):
        raise ValueError(
            "I nomi delle feature non corrispondono "
            "alle matrici trasformate."
        )

    # --------------------------------------------------------
    # 5. CALCOLO SHAP - PARTE 7.3
    # --------------------------------------------------------

    explainer, shap_values, local_idx = (
        compute_shap_values(
            fitted_pipeline=fitted_pipeline,
            X_transformed=X_explain,
            X_background_transformed=X_background,
            max_background=max_background,
            max_explain=max_explain,
            random_state=random_state,
            return_indices=True
        )
    )

    # Gli indici sono relativi al campione X_explain.
    explained_ids = (
        explained_ids.iloc[local_idx]
        .reset_index(drop=True)
    )

    if len(explained_ids) != len(shap_values.values):
        raise ValueError(
            "Identificativi e valori SHAP "
            "non sono allineati."
        )

    # --------------------------------------------------------
    # 6. IMPORTANZA GLOBALE E SPIEGAZIONE LOCALE
    # --------------------------------------------------------

    global_importance = summarize_global_shap_importance(
        shap_values=shap_values,
        feature_names=feature_names
    )

    local_explanation = get_local_shap_explanation(
        shap_values=shap_values,
        feature_names=feature_names,
        observation_index=0
    )

    # --------------------------------------------------------
    # 7. SALVATAGGIO PRINCIPALE - PARTE 7.6
    # --------------------------------------------------------

    paths = save_shap_artifacts(
        base_dir=Path(base_dir),
        project=project,
        model_name=model_name,
        configuration=configuration,
        shap_values=shap_values,
        feature_names=feature_names,
        global_importance=global_importance,
        snapshot=snapshot,
        scenario=scenario,
        task=task,
        threshold=threshold,
        balancing=balancing,
        local_explanation=local_explanation,
        explained_ids=explained_ids
    )

    # --------------------------------------------------------
    # 8. SALVATAGGIO DEL BACKGROUND
    # --------------------------------------------------------

    output_dir = Path(
        paths["global_importance_path"]
    ).parent

    background_ids_path = (
        output_dir
        / "background_ids.csv"
    )

    background_data_path = (
        output_dir
        / "background_transformed.npy"
    )

    background_ids.to_csv(
        background_ids_path,
        index=False
    )

    np.save(
        background_data_path,
        np.asarray(X_background)
    )

    # --------------------------------------------------------
    # 9. METADATI DELLE SPIEGAZIONI
    # --------------------------------------------------------

    metadata = {
        "scenario": scenario,
        "snapshot": snapshot,
        "project": project,
        "task": task,
        "model_name": model_name,
        "configuration": configuration,
        "threshold_label": threshold,
        "balancing": balancing,
        "random_state": random_state,
        "n_background": len(X_background),
        "n_explained": len(explained_ids),
        "n_features": len(feature_names),
        "max_background": max_background,
        "max_explain": max_explain,
        "shap_version": shap.__version__,
        "regression_output": (
            "Raw model prediction; "
            "evaluation clips negative values to zero"
        ),
        "classifier_output": (
            "SLOW probability when available; "
            "otherwise decision score"
        )
    }

    metadata_path = (
        output_dir
        / "shap_metadata.json"
    )

    metadata_path.write_text(
        json.dumps(
            metadata,
            indent=2,
            default=str
        ),
        encoding="utf-8"
    )

    # --------------------------------------------------------
    # 10. AGGIORNAMENTO DEI PERCORSI
    # --------------------------------------------------------

    paths["background_ids_path"] = (
        background_ids_path
    )

    paths["background_data_path"] = (
        background_data_path
    )

    paths["metadata_path"] = metadata_path

    # --------------------------------------------------------
    # 11. OUTPUT
    # --------------------------------------------------------

    return {
        "explainer": explainer,
        "shap_values": shap_values,
        "global_importance": global_importance,
        "local_explanation": local_explanation,
        "explained_ids": explained_ids,
        "background_ids": background_ids,
        "paths": paths
    }


print(
    "Orchestrazione SHAP multi-scenario aggiornata "
    "con metadati e background."
)


Orchestrazione SHAP multi-scenario aggiornata con metadati e background.


**Grafici SHAP**

In [25]:

# ============================================================
# 9.3 GENERAZIONE E SALVATAGGIO DEI GRAFICI SHAP
# BAR PLOT - BEESWARM - DEPENDENCE PLOT
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap


def save_scenario_shap_plots(
    shap_result,
    max_display=15,
    dependence_top_n=3
):
    """
    Genera e salva i grafici SHAP di un esperimento
    già eseguito tramite run_scenario_shap().

    I nomi delle feature vengono recuperati
    dal CSV prodotto dalla funzione di salvataggio.
    """

    # --------------------------------------------------------
    # 1. RECUPERO DEI RISULTATI
    # --------------------------------------------------------

    original_shap = shap_result["shap_values"]

    feature_path = Path(
        shap_result["paths"]["feature_names_path"]
    )

    global_path = Path(
        shap_result["paths"]["global_importance_path"]
    )

    if not feature_path.is_file():
        raise FileNotFoundError(
            f"File delle feature mancante: {feature_path}"
        )

    feature_names = pd.read_csv(
        feature_path
    )["feature"].tolist()

    values = np.asarray(original_shap.values)

    if values.ndim != 2:
        raise ValueError(
            "I valori SHAP devono essere bidimensionali."
        )

    if len(feature_names) != values.shape[1]:
        raise ValueError(
            "Numero di nomi feature non coerente "
            "con la matrice SHAP."
        )

    # --------------------------------------------------------
    # 2. PREPARAZIONE DELL'OGGETTO SHAP
    # --------------------------------------------------------

    shap_values = shap.Explanation(
        values=values,
        base_values=original_shap.base_values,
        data=original_shap.data,
        feature_names=feature_names
    )

    global_importance = shap_result[
        "global_importance"
    ]

    # --------------------------------------------------------
    # 3. CARTELLA E PERCORSI
    # --------------------------------------------------------

    output_dir = global_path.parent

    if not output_dir.is_dir():
        raise FileNotFoundError(
            f"Cartella SHAP mancante: {output_dir}"
        )

    plot_paths = {
        "bar": output_dir / "shap_bar.png",
        "beeswarm": output_dir / "shap_beeswarm.png"
    }

    top_features = (
        global_importance["feature"]
        .head(dependence_top_n)
        .tolist()
    )

    for i, feature in enumerate(
        top_features,
        start=1
    ):
        plot_paths[f"dependence_{i}"] = (
            output_dir
            / f"shap_dependence_{i}.png"
        )

    # Non sovrascriviamo grafici esistenti
    existing_paths = [
        path
        for path in plot_paths.values()
        if path.exists()
    ]

    if existing_paths:
        raise FileExistsError(
            "Uno o più grafici SHAP esistono già: "
            + ", ".join(map(str, existing_paths))
        )

    # --------------------------------------------------------
    # 4. BAR PLOT
    # --------------------------------------------------------

    try:
        shap.plots.bar(
            shap_values,
            max_display=max_display,
            show=False
        )

        plt.savefig(
            plot_paths["bar"],
            dpi=200,
            bbox_inches="tight"
        )

    finally:
        plt.close("all")

    # --------------------------------------------------------
    # 5. BEESWARM
    # --------------------------------------------------------

    try:
        shap.plots.beeswarm(
            shap_values,
            max_display=max_display,
            show=False
        )

        plt.savefig(
            plot_paths["beeswarm"],
            dpi=200,
            bbox_inches="tight"
        )

    finally:
        plt.close("all")

    # --------------------------------------------------------
    # 6. DEPENDENCE PLOT
    # --------------------------------------------------------

    for i, feature in enumerate(
        top_features,
        start=1
    ):

        try:
            shap.plots.scatter(
                shap_values[:, feature],
                show=False
            )

            plt.savefig(
                plot_paths[f"dependence_{i}"],
                dpi=200,
                bbox_inches="tight"
            )

        finally:
            plt.close("all")

    return plot_paths


print(
    "Funzione di generazione dei grafici SHAP "
    "configurata con nomi delle feature allineati."
)


Funzione di generazione dei grafici SHAP configurata con nomi delle feature allineati.


**Confronto risultati SHAP**

In [26]:

# ============================================================
# 9.4 CONFRONTO SHAP TRA ESPERIMENTI
# RIUTILIZZO DELLE FUNZIONI DELLA PARTE 7
# ============================================================

from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd


def compare_saved_shap_experiments(
    importance_paths
):
    """
    Confronta ranking SHAP salvati in esperimenti diversi.

    importance_paths:
        dizionario nel formato:
        {
            "nome_esperimento": path_csv,
            ...
        }

    Restituisce:
        - tabelle delle importanze caricate
        - confronti pairwise
        - matrici Spearman/Jaccard
    """

    if len(importance_paths) < 2:
        raise ValueError(
            "Servono almeno due esperimenti SHAP."
        )

    # --------------------------------------------------------
    # 1. CARICAMENTO DELLE IMPORTANZE
    # --------------------------------------------------------

    importance_tables = {}

    for experiment_name, path in importance_paths.items():

        path = Path(path)

        if not path.is_file():
            raise FileNotFoundError(
                f"Risultato SHAP mancante: {path}"
            )

        df = pd.read_csv(path)

        required_columns = [
            "feature",
            "mean_abs_shap"
        ]

        if not all(
            col in df.columns
            for col in required_columns
        ):
            raise ValueError(
                f"Colonne SHAP mancanti in {path}"
            )

        if (
            df["feature"].isna().any()
            or df["feature"].duplicated().any()
            or df["mean_abs_shap"].isna().any()
            or not np.isfinite(
                df["mean_abs_shap"].to_numpy(
                    dtype=float
                )
            ).all()
        ):
            raise ValueError(
                f"Importanze SHAP non valide: {path}"
            )

        importance_tables[experiment_name] = (
            df[required_columns].copy()
        )

    # --------------------------------------------------------
    # 2. CONFRONTI A COPPIE
    # --------------------------------------------------------

    comparison_rows = []

    for name_a, name_b in combinations(
        importance_tables.keys(),
        2
    ):

        importance_a = importance_tables[name_a]
        importance_b = importance_tables[name_b]

        features_a = set(importance_a["feature"])
        features_b = set(importance_b["feature"])

        common_features = features_a & features_b
        union_features = features_a | features_b

        comparison = compare_shap_rankings(
            importance_a=importance_a,
            importance_b=importance_b,
            top_k_values=(5, 10)
        )

        comparison_rows.append({
            "experiment_a": name_a,
            "experiment_b": name_b,
            "n_features_a": len(features_a),
            "n_features_b": len(features_b),
            "n_common_features": len(common_features),
            "feature_overlap": (
                len(common_features) / len(union_features)
                if union_features else np.nan
            ),
            **comparison
        })

    comparison_df = pd.DataFrame(
        comparison_rows
    )

    # --------------------------------------------------------
    # 3. MATRICI DI STABILITÀ
    # --------------------------------------------------------

    stability_matrices = build_shap_stability_matrices(
        importance_tables
    )

    # --------------------------------------------------------
    # 4. RISULTATI
    # --------------------------------------------------------

    return {
        "importance_tables": importance_tables,
        "comparisons": comparison_df,
        "stability_matrices": stability_matrices
    }


print(
    "Funzione di confronto SHAP tra esperimenti configurata."
)


Funzione di confronto SHAP tra esperimenti configurata.


**Output SHAP**

In [27]:

# ============================================================
# 9.6 ORCHESTRAZIONE DEGLI OUTPUT SHAP
# GRAFICI SHAP
# ============================================================

from pathlib import Path
import pandas as pd


def complete_scenario_shap_outputs(
    shap_result,
    snapshot,
    project=None,
    generate_plots=True,
    max_display=15,
    dependence_top_n=3
):
    """
    Completa gli output di un esperimento SHAP
    già eseguito tramite run_scenario_shap().

    Produce:
        - grafici SHAP, se richiesti

    Non ricalcola i valori SHAP.
    """

    # --------------------------------------------------------
    # 1. RECUPERO DEI RISULTATI
    # --------------------------------------------------------

    global_importance = shap_result[
        "global_importance"
    ]

    output_dir = Path(
        shap_result["paths"][
            "global_importance_path"
        ]
    ).parent

    if not output_dir.is_dir():
        raise FileNotFoundError(
            f"Cartella SHAP inesistente: {output_dir}"
        )

    output_paths = dict(shap_result["paths"])

    # --------------------------------------------------------
    # 2. GENERAZIONE DEI GRAFICI
    # --------------------------------------------------------

    if generate_plots:

        plot_paths = save_scenario_shap_plots(
            shap_result=shap_result,
            max_display=max_display,
            dependence_top_n=dependence_top_n
        )

        output_paths.update({
            f"plot_{name}": path
            for name, path in plot_paths.items()
        })

    # --------------------------------------------------------
    # 3. RESTITUZIONE
    # --------------------------------------------------------

    return {
        "output_paths": output_paths,
        "output_dir": output_dir
    }


print(
    "Orchestrazione degli output SHAP configurata."
)


Orchestrazione degli output SHAP configurata.


**Esecuzione e verifica grafici**

In [28]:

# ============================================================
# 9.7 ORCHESTRAZIONE COMPLETA XAI PER ESPERIMENTO
# CALCOLO SHAP + SALVATAGGIO + GRAFICI
# ============================================================

from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
from sklearn.dummy import DummyRegressor


def run_complete_scenario_xai(
    fitted_pipeline,
    scenario_data,
    base_dir,
    scenario,
    snapshot,
    project,
    task,
    model_name,
    configuration,
    threshold=None,
    balancing=None,
    max_background=50,
    max_explain=30,
    random_state=42,
    generate_plots=True,
    max_display=15,
    dependence_top_n=3
):
    """
    Esegue il flusso XAI per un modello già addestrato.

    Riutilizza:
        - 9.1: calcolo e salvataggio SHAP
        - 9.6: grafici

    Non ripete training, preprocessing o valutazione.
    """

    # --------------------------------------------------------
    # 1. CALCOLO E SALVATAGGIO SHAP
    # --------------------------------------------------------

    shap_result = run_scenario_shap(
        fitted_pipeline=fitted_pipeline,
        scenario_data=scenario_data,
        base_dir=base_dir,
        scenario=scenario,
        snapshot=snapshot,
        project=project,
        task=task,
        model_name=model_name,
        configuration=configuration,
        threshold=threshold,
        balancing=balancing,
        max_background=max_background,
        max_explain=max_explain,
        random_state=random_state
    )

    # --------------------------------------------------------
    # 2. COMPLETAMENTO DEGLI OUTPUT
    # --------------------------------------------------------

    completed = complete_scenario_shap_outputs(
        shap_result=shap_result,
        snapshot=snapshot,
        project=project,
        generate_plots=generate_plots,
        max_display=max_display,
        dependence_top_n=dependence_top_n
    )

    # --------------------------------------------------------
    # 3. CONTROLLO DEI FILE PRODOTTI
    # --------------------------------------------------------

    output_paths = completed["output_paths"]

    saved_files_valid = all(
        path is None or Path(path).is_file()
        for path in output_paths.values()
    )

    if not saved_files_valid:
        raise RuntimeError(
            "Alcuni output XAI non sono stati salvati."
        )

    # --------------------------------------------------------
    # 4. RESTITUZIONE DEI RISULTATI
    # --------------------------------------------------------

    return {
        "shap_result": shap_result,
        "output_paths": output_paths,
        "output_dir": completed["output_dir"],
        "n_explained": len(
            shap_result["explained_ids"]
        )
    }


print("Orchestrazione XAI completa configurata.")

Orchestrazione XAI completa configurata.


### **12 Esperimenti**

In [29]:
# CONFIGURAZIONE SHAP

SCENARIO = "within_project"  # "within_project", "pooled" oppure "lopo"
PROGETTI_SCENARIO = ["KDE"]  # Progetti da valutare o da escludere a turno.
SNAPSHOT_SCENARIO = ["T0"]   # Usa list(VALID_SNAPSHOTS) per tutti gli snapshot.
SOGLIE_SCENARIO = {"P50": 0.50}  # Aggiungi "P75": 0.75 e "P90": 0.90 per la griglia completa.
METODO_BILANCIAMENTO = "oversampled"  # "natural" oppure "oversampled"

if SCENARIO not in {"within_project", "pooled", "lopo"}:
    raise ValueError(f"Scenario sconosciuto: {SCENARIO}")
if METODO_BILANCIAMENTO not in {"natural", "oversampled"}:
    raise ValueError(f"Bilanciamento sconosciuto: {METODO_BILANCIAMENTO}")
if SCENARIO != "pooled" and (
    not PROGETTI_SCENARIO or set(PROGETTI_SCENARIO) - set(PROJECTS)
):
    raise ValueError("Indica uno o più progetti presenti in PROJECTS.")

# Pooled usa tutti i progetti di PROJECTS con un singolo modello.
SELEZIONE_PROGETTI = [None] if SCENARIO == "pooled" else list(dict.fromkeys(PROGETTI_SCENARIO))


**SHAP**

In [30]:
# SHAP SUI CLASSIFICATORI DELLO SCENARIO SELEZIONATO

ESEGUI_SHAP = False

if ESEGUI_SHAP:
    import json
    from itertools import product
    from pathlib import Path
    from tempfile import TemporaryDirectory
    from time import perf_counter

    import joblib

    # Vuoto = cerca in tutti i run. In caso di duplicati, elenca i run da usare.
    cartelle_run_shap = []
    max_background = 20
    max_explain = 10

    risultati = BASE / "results"
    cartelle_modelli = (
        [risultati / nome / SCENARIO for nome in cartelle_run_shap]
        if cartelle_run_shap
        else sorted(risultati.glob(f"*/{SCENARIO}"))
    )
    sigle_modelli = {"RandomForest": "rf", "SVM": "svm"}
    etichette_soglie = {q: nome for nome, q in SOGLIE_SCENARIO.items()}
    nomi_progetti = {
        (progetto if progetto is not None else "ALL_PROJECTS"): progetto
        for progetto in SELEZIONE_PROGETTI
    }
    griglia = list(product(nomi_progetti, SNAPSHOT_SCENARIO, etichette_soglie, sigle_modelli))
    richiesti = set(griglia)
    modelli_salvati = {}
    resampler_atteso = (
        "RandomOverSampler(random_state=42)"
        if METODO_BILANCIAMENTO == "oversampled"
        else None
    )

    for cartella in cartelle_modelli:
        for file_meta in sorted(cartella.glob("*_metadata.json")):
            try:
                meta = json.loads(file_meta.read_text(encoding="utf-8"))
            except (OSError, ValueError):
                continue

            chiave = (
                meta.get("project"), meta.get("snapshot"), meta.get("quantile"),
                meta.get("model_name"),
            )
            if (
                chiave not in richiesti
                or meta.get("scenario") != SCENARIO
                or meta.get("task") != "classification"
                or meta.get("configuration") != "default"
                or meta.get("balancing") != METODO_BILANCIAMENTO
                or (meta.get("parameters") or {}).get("resampler") != resampler_atteso
            ):
                continue

            nome = file_meta.name.removesuffix("_metadata.json")
            percorso_modello = cartella / f"{nome}_model.joblib"
            percorso_predizioni = cartella / f"{nome}_predictions.csv"
            if not percorso_modello.is_file() or not percorso_predizioni.is_file():
                continue

            if chiave in modelli_salvati:
                precedente = modelli_salvati[chiave][0].parent.parent.name
                raise ValueError(
                    f"Modello duplicato per {chiave}: {precedente}, {cartella.parent.name}. "
                    "Imposta cartelle_run_shap con i run da usare."
                )

            modelli_salvati[chiave] = (percorso_modello, percorso_predizioni)

    mancanti = [chiave for chiave in griglia if chiave not in modelli_salvati]
    if mancanti:
        raise FileNotFoundError(
            f"Mancano {len(mancanti)} modelli {SCENARIO}/{METODO_BILANCIAMENTO}. "
            f"Prime configurazioni mancanti: {mancanti[:6]}"
        )

    dati_caricati = None
    cartella_shap = risultati / "shap"
    if SCENARIO != "within_project":
        cartella_shap = cartella_shap / SCENARIO

    for numero, chiave in enumerate(griglia, start=1):
        nome_progetto, snapshot, quantile, nome_modello = chiave
        progetto = nomi_progetti[nome_progetto]
        soglia = etichette_soglie[quantile]
        sigla = sigle_modelli[nome_modello]
        suffisso = "" if METODO_BILANCIAMENTO == "natural" else f"_{METODO_BILANCIAMENTO}"
        radice_shap = cartella_shap / f"{nome_progetto}_SHAP{suffisso}"
        cartella_modello = radice_shap / f"{nome_progetto}_SHAP_{sigla}"
        destinazione = cartella_modello / f"{snapshot}_{soglia}"
        etichetta = f"[{numero}/{len(griglia)}] {nome_progetto} | {snapshot} | {soglia} | {nome_modello}"

        if (destinazione / "shap_metadata.json").is_file():
            print(f"{etichetta} | già salvato", flush=True)
            continue
        if destinazione.exists():
            raise FileExistsError(f"Output SHAP incompleto già presente: {destinazione}")

        if dati_caricati != (progetto, snapshot):
            dati_shap = prepare_scenario_data(
                base_dir=BASE, scenario=SCENARIO, snapshot=snapshot,
                projects=PROJECTS, project=progetto,
            )
            dati_caricati = (progetto, snapshot)

        percorso_modello, percorso_predizioni = modelli_salvati[chiave]
        predizioni = pd.read_csv(
            percorso_predizioni, usecols=["project", "bug_id"], dtype=str,
        )
        ids_test = dati_shap["ids_test"][["project", "bug_id"]].astype(str).reset_index(drop=True)
        if not predizioni.equals(ids_test):
            raise ValueError(f"Il test ricostruito differisce da quello salvato: {chiave}")

        modello = joblib.load(percorso_modello)
        print(etichetta, flush=True)
        inizio = perf_counter()
        cartella_modello.mkdir(parents=True, exist_ok=True)

        with TemporaryDirectory(dir=cartella_shap, prefix="xai_") as temporanea:
            risultato = run_complete_scenario_xai(
                fitted_pipeline=modello,
                scenario_data=dati_shap,
                base_dir=Path(temporanea),
                scenario=SCENARIO,
                snapshot=snapshot,
                project=progetto,
                task="classification",
                model_name=nome_modello,
                configuration="default",
                threshold=soglia,
                balancing=METODO_BILANCIAMENTO,
                max_background=max_background,
                max_explain=max_explain,
                random_state=42,
                generate_plots=True,
            )
            Path(risultato["output_dir"]).rename(destinazione)

        print(f"Salvato in {perf_counter() - inizio:.1f} s | {destinazione}", flush=True)
